In [102]:
!pip install PyPortfolioOpt
!pip install scikit-learn
!pip install stable-baselines3
!pip install gymnasium
!pip install pypfopt
!pip install tqdm
!pip install torch
!pip install torchvision

ERROR: Could not find a version that satisfies the requirement pypfopt (from versions: none)
ERROR: No matching distribution found for pypfopt


In [103]:
import numpy as np
import pandas as pd
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import A2C
from stable_baselines3.common.vec_env import DummyVecEnv
from sklearn.preprocessing import StandardScaler
import torch
from datetime import datetime
from stable_baselines3.common.noise import NormalActionNoise

In [104]:
class StockPortfolioIterator:
    def __init__(self, file_path: str, lookback_window: int = 30, min_history: int = 100, max_stocks: int = 100, train_years: int = 20):

        print(f"Loading data from {file_path}...")
        self.data = pd.read_csv(file_path)
        self.data['date'] = pd.to_datetime(self.data['date'])

        # Only use recent data (last few years)
        cutoff_date = self.data['date'].max() - pd.DateOffset(years=train_years)
        self.data = self.data[self.data['date'] >= cutoff_date]

        self.data = self.data.sort_values(['date', 'tic'])

        # Filter stocks with enough history
        stock_counts = self.data.groupby('tic').size()
        valid_stocks = stock_counts[stock_counts >= min_history].index

        # Select stocks with highest trading volume (more likely to be liquid)
        if len(valid_stocks) > max_stocks:
            avg_volumes = self.data[self.data['tic'].isin(valid_stocks)].groupby('tic')['volume'].mean()
            valid_stocks = avg_volumes.nlargest(max_stocks).index.tolist()

        self.data = self.data[self.data['tic'].isin(valid_stocks)]
        self.unique_dates = sorted(self.data['date'].unique())
        self.tickers = sorted(self.data['tic'].unique())
        self.num_stocks = len(self.tickers)
        self.lookback = lookback_window
        self.num_features = 7

        # Create ticker lookup for faster access
        self.ticker_indices = {ticker: i for i, ticker in enumerate(self.tickers)}

        self._add_technical_indicators()
        self._scale_features()

        # Store data in a more efficient format
        self.prepare_training_data()

        print(f"Loaded {self.num_stocks} stocks with {len(self.unique_dates)} trading days")
        print(f"Date range: {self.unique_dates[0]} to {self.unique_dates[-1]}")

    def _add_technical_indicators(self):

        print("Adding technical indicators...")
        indicators = ['Returns', 'Volume_MA', 'Price_MA', 'RSI', 'MACD', 'Volatility', 'Momentum']

        # Process one ticker at a time to avoid memory issues
        for tic in self.tickers:
            mask = self.data['tic'] == tic
            stock_data = self.data.loc[mask].copy()

            # Calculate technical indicators
            stock_data['Returns'] = stock_data['close'].pct_change()
            stock_data['Price_MA'] = stock_data['close'].rolling(window=20, min_periods=1).mean()
            stock_data['Momentum'] = stock_data['close'].pct_change(periods=10)
            stock_data['Volume_MA'] = stock_data['volume'].rolling(window=10, min_periods=1).mean()

            # RSI Calculation
            delta = stock_data['close'].diff()
            gain = (delta.clip(lower=0)).rolling(window=14, min_periods=1).mean()
            loss = (-delta.clip(upper=0)).rolling(window=14, min_periods=1).mean()
            rs = gain / (loss + 1e-8)
            stock_data['RSI'] = 100 - (100 / (1 + rs))

            # MACD Calculation
            exp1 = stock_data['close'].ewm(span=12, adjust=False).mean()
            exp2 = stock_data['close'].ewm(span=26, adjust=False).mean()
            stock_data['MACD'] = exp1 - exp2

            # Volatility
            stock_data['Volatility'] = stock_data['Returns'].rolling(window=30, min_periods=1).std()

            # Update the main dataframe
            self.data.loc[mask, indicators] = stock_data[indicators]

        # Fill missing values
        self.data = self.data.fillna(0)
        print("Technical indicators added successfully")

    def _scale_features(self):
        print("Scaling features...")
        feature_columns = ['Returns', 'Volume_MA', 'Price_MA', 'RSI', 'MACD', 'Volatility', 'Momentum']

        for tic in self.tickers:
            mask = self.data['tic'] == tic
            scaler = StandardScaler()
            self.data.loc[mask, feature_columns] = scaler.fit_transform(self.data.loc[mask, feature_columns])

    def prepare_training_data(self):
        print("Preparing training data...")
        # Pre-allocate arrays for features and prices
        self.all_features = {}
        self.all_prices = {}
        feature_columns = ['Returns', 'Volume_MA', 'Price_MA', 'RSI', 'MACD', 'Volatility', 'Momentum']

        # Process data by date for efficient access during training
        for date_idx, date in enumerate(self.unique_dates):
            date_mask = self.data['date'] == date
            date_data = self.data[date_mask]

            features = np.zeros((self.num_stocks, self.num_features))
            prices = np.zeros(self.num_stocks)

            for _, row in date_data.iterrows():
                ticker_idx = self.ticker_indices.get(row['tic'])
                if ticker_idx is not None:
                    features[ticker_idx] = row[feature_columns].values
                    prices[ticker_idx] = row['close']

            self.all_features[date] = features
            self.all_prices[date] = prices

    def __iter__(self):
        """Initialize iterator"""
        self.current_idx = self.lookback
        return self

    def __next__(self):
        """Get next batch of data"""
        if self.current_idx >= len(self.unique_dates):
            raise StopIteration

        current_date = self.unique_dates[self.current_idx]
        window_dates = self.unique_dates[self.current_idx - self.lookback:self.current_idx]

        # Get lookback window features
        features = np.zeros((self.num_stocks, self.lookback, self.num_features), dtype=np.float32)

        for i, date in enumerate(window_dates):
            if date in self.all_features:
                features[:, i, :] = self.all_features[date]

        prices = self.all_prices[current_date]

        self.current_idx += 1
        return {
            'features': features,
            'prices': prices,
            'date': current_date,
            'tickers': self.tickers
        }

    def reset(self):
        """Reset the iterator"""
        self.current_idx = self.lookback
        return self

In [105]:
class PortfolioRecommender:
    def __init__(self, model_path, data_path, max_stocks=100, lookback=30):
        """Initialize the portfolio recommender with the trained A2C model and latest stock data."""
        self.model = A2C.load(model_path)  # Load A2C model
        self.max_stocks = max_stocks
        self.lookback = lookback

        # Load and preprocess stock data
        data = pd.read_csv(data_path)
        data['date'] = pd.to_datetime(data['date'])

        # Get the most recent trading date
        self.latest_date = data['date'].max()

        # Select top stocks by trading volume (same filtering as in training)
        recent_data = data[data['date'] >= (self.latest_date - pd.DateOffset(days=30))]
        avg_volumes = recent_data.groupby('tic')['volume'].mean()
        top_tickers = avg_volumes.nlargest(max_stocks).index.tolist()

        # Get the ordered list of tickers (must match training data order)
        self.tickers = sorted(top_tickers)

        # Store latest closing prices for selected stocks
        latest_data = data[data['date'] == self.latest_date]
        self.latest_prices = {row['tic']: row['close'] for _, row in latest_data.iterrows()
                              if row['tic'] in self.tickers}

        # Compute technical indicators (ensures feature consistency)
        self._prepare_features(data)

    def _prepare_features(self, data):
        """Prepares feature matrix for the latest available stock data."""
        filtered_data = data[data['tic'].isin(self.tickers)]

        # Get the most recent dates for feature calculation
        recent_dates = sorted(filtered_data['date'].unique())[-self.lookback:]
        recent_data = filtered_data[filtered_data['date'].isin(recent_dates)]

        # Initialize the feature array
        features = np.zeros((self.max_stocks, self.lookback, 7), dtype=np.float32)

        for i, ticker in enumerate(self.tickers):
            if i >= self.max_stocks:
                break

            ticker_data = recent_data[recent_data['tic'] == ticker].sort_values('date')
            if len(ticker_data) == self.lookback:
                # Create feature array (must match training data features)
                ticker_features = np.zeros((self.lookback, 7))
                ticker_features[:, 0] = ticker_data['close'].pct_change().fillna(0).values  # Returns
                ticker_features[:, 1] = ticker_data['volume'].values / ticker_data['volume'].mean()  # Normalized Volume
                ticker_features[:, 2] = ticker_data['close'].values / ticker_data['close'].mean()  # Normalized Price
                ticker_features[:, 3] = 50  # Placeholder for RSI
                ticker_features[:, 4] = 0   # Placeholder for MACD
                ticker_features[:, 5] = ticker_data['close'].pct_change().rolling(5).std().fillna(0).values  # Volatility
                ticker_features[:, 6] = ticker_data['close'].pct_change(5).fillna(0).values  # Momentum

                features[i] = ticker_features

        self.recent_features = features

    def recommend_portfolio(self, amount_cad):
        """Generates stock allocation recommendations based on the trained A2C model."""

        # Use the latest computed features for prediction
        action, _ = self.model.predict(self.recent_features, deterministic=True)

        # Normalize action values to sum to 1
        action = np.clip(action, 0, 1)
        if action.sum() > 0:
            action /= action.sum()

        # Allocate cash and stocks
        allocations = {}
        cash_allocation = action[0] * amount_cad  # First action is cash allocation

        for i, ticker in enumerate(self.tickers):
            if i >= len(self.tickers) or i+1 >= len(action):
                continue

            allocation = action[i+1] * amount_cad  # Skip first index (cash)
            if allocation > 0 and ticker in self.latest_prices:
                price = self.latest_prices[ticker]
                shares = allocation / price if price > 0 else 0

                allocations[ticker] = {
                    "allocation_percent": float(action[i+1] * 100),
                    "allocation_cad": float(allocation),
                    "price_per_share": float(price),
                    "shares": float(shares)
                }

        return {
            "cash_percent": float(action[0] * 100),
            "cash_amount": float(cash_allocation),
            "stock_allocations": allocations,
            "total_amount": float(amount_cad),
            "recommendation_date": str(self.latest_date)
        }


In [106]:
class PortfolioTradingEnv(gym.Env):
    def __init__(self, data_iterator, initial_cash=10000, transaction_cost=0.001):
        super().__init__()
        self.data_iterator = data_iterator
        self.initial_cash = initial_cash
        self.transaction_cost = transaction_cost
        self.num_stocks = data_iterator.num_stocks
        self.lookback = data_iterator.lookback
        self.num_features = data_iterator.num_features

        # Action space for continuous portfolio allocation
        self.action_space = spaces.Box(
            low=0, high=1, shape=(self.num_stocks + 1,), dtype=np.float32
        )

        # Observation space: stock features over the lookback period
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf,
            shape=(self.num_stocks, self.lookback, self.num_features), dtype=np.float32
        )

        self.reset()

    def reset(self, seed=None, options=None):
        """Reset the environment at the beginning of an episode"""
        self.data_iterator.reset()
        self.cash = self.initial_cash
        self.holdings = np.zeros(self.num_stocks)

        try:
            self.current_step = next(self.data_iterator)
            return self._get_observation(), {}
        except StopIteration:
            return np.zeros((self.num_stocks, self.lookback, self.num_features), dtype=np.float32), {}

    def _get_observation(self):
        return self.current_step['features'].astype(np.float32)

    def step(self, action):
        """Execute a trade based on action provided by A2C model"""
        action = np.clip(action, 0, 1)
        action_sum = action.sum()
        if action_sum > 0:
            action /= action_sum

        current_prices = self.current_step['prices']
        portfolio_value = self.cash + np.sum(self.holdings * current_prices)

        target_values = action[1:] * portfolio_value
        current_values = self.holdings * current_prices
        delta_values = target_values - current_values

        transaction_costs = np.abs(delta_values).sum() * self.transaction_cost

        for i in range(self.num_stocks):
            if current_prices[i] > 0:
                self.holdings[i] += delta_values[i] / current_prices[i]

        self.cash = portfolio_value - np.sum(self.holdings * current_prices) - transaction_costs
        old_portfolio_value = portfolio_value

        try:
            self.current_step = next(self.data_iterator)
            new_prices = self.current_step['prices']
            done = False
        except StopIteration:
            new_prices = current_prices
            done = True

        new_portfolio_value = self.cash + np.sum(self.holdings * new_prices)
        reward = (new_portfolio_value / old_portfolio_value) - 1 - (transaction_costs / old_portfolio_value)

        return (
            self._get_observation() if not done else np.zeros_like(self.observation_space.sample()),
            float(reward),
            done,
            False,
            {"portfolio_value": new_portfolio_value}
        )

# model evaluation

In [107]:
def evaluate_model(model, data_path, max_stocks=100, lookback=30, episodes=10):
    """Evaluate the model on a validation dataset"""
    print(f"Evaluating model performance...")

    eval_iter = StockPortfolioIterator(
        data_path,
        lookback_window=lookback,
        min_history=100,
        max_stocks=max_stocks,
        train_years=5  # Use more recent data for evaluation
    )

    eval_env = PortfolioTradingEnv(eval_iter)

    total_rewards = []
    portfolio_values = []

    for ep in range(episodes):
        obs, _ = eval_env.reset()
        done = False
        episode_reward = 0
        initial_value = eval_env.initial_cash
        step_count = 0

        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, done, _, info = eval_env.step(action)
            episode_reward += reward
            step_count += 1

            if done:
                final_value = info["portfolio_value"]
                portfolio_values.append(final_value)

                # Calculate annualized return
                if step_count > 0:
                    days = step_count
                    annualized_return = ((final_value / initial_value) ** (252 / days) - 1) * 100
                else:
                    annualized_return = 0

                print(f"Episode {ep+1}: Return = {episode_reward:.4f}, "
                      f"Steps = {step_count}, "
                      f"Final Value = ${final_value:.2f}, "
                      f"Annualized Return = {annualized_return:.2f}%")

        total_rewards.append(episode_reward)

    # Calculate summary statistics
    avg_reward = np.mean(total_rewards)
    avg_final_value = np.mean(portfolio_values)
    success_rate = np.sum(np.array(total_rewards) > 0) / len(total_rewards) * 100

    print(f"\nEvaluation Results (over {episodes} episodes):")
    print(f"Average Total Reward: {avg_reward:.4f}")
    print(f"Average Final Portfolio Value: ${avg_final_value:.2f}")
    print(f"Success Rate (positive return): {success_rate:.2f}%")

    return total_rewards, portfolio_values

# Training the A2c model

In [108]:


def train_model(data_path, model_save_path, max_stocks=100, lookback=30, timesteps=500000):
    print(f"Starting training at {datetime.now()}")

    # Initialize components with more manageable parameters
    train_iter = StockPortfolioIterator(
        data_path,
        lookback_window=lookback,
        min_history=100,
        max_stocks=max_stocks,
        train_years=20
    )

    def make_env():
        return PortfolioTradingEnv(train_iter)

    train_env = DummyVecEnv([make_env])

    # Configure A2C model with suitable hyperparameters
    n_actions = train_iter.num_stocks + 1  # Action space size
    action_noise = NormalActionNoise(mean=np.zeros(n_actions), sigma=0.1 * np.ones(n_actions))
    model = A2C(
      policy="MlpPolicy",
      env=train_env  # Make sure `train_env` is defined earlier in your code
    )



    # Log initial random policy performance
    print("Evaluating random policy before training...")
    try:
        obs = train_env.reset()
        done = np.array([False])
        total_reward = 0
        step_count = 0
        max_steps = 100  # Limit evaluation to avoid infinite loops

        while not done.any() and step_count < max_steps:
            actions = []
            for i in range(train_env.num_envs):
                action = np.random.random(train_iter.num_stocks + 1)
                action = action / action.sum()
                actions.append(action)

            actions = np.array(actions)
            obs, rewards, done, info = train_env.step(actions)
            total_reward += rewards[0]
            step_count += 1

        print(f"Random policy evaluation: Total steps: {step_count}, Total reward: {total_reward}")
    except Exception as e:
        print(f"Error during random policy evaluation: {e}")
        print("Continuing with training anyway...")

    # Start training
    print(f"Training model with {train_iter.num_stocks} stocks, each with {lookback} days of history")

    stage_timesteps = timesteps // 5
    for stage in range(5):
        print(f"\nTraining stage {stage+1}/5 ({stage_timesteps} timesteps)...")
        try:
            model.learn(total_timesteps=stage_timesteps, progress_bar=True, reset_num_timesteps=False)

            # Save checkpoint
            checkpoint_path = f"{model_save_path}_checkpoint_{stage+1}"
            model.save(checkpoint_path)
            print(f"Checkpoint saved to {checkpoint_path}")

            # Quick evaluation
            if stage < 4:  # Skip final evaluation as we'll do a full one later
                print("Quick evaluation of current policy...")
                eval_env = make_env()
                for i in range(3):
                    obs, _ = eval_env.reset()
                    done = False
                    episode_reward = 0
                    step_count = 0
                    max_eval_steps = 100

                    while not done and step_count < max_eval_steps:
                        action, _ = model.predict(obs, deterministic=True)
                        obs, reward, done, _, info = eval_env.step(action)
                        episode_reward += reward
                        step_count += 1

                    print(f"  Episode {i+1} reward: {episode_reward:.4f}, Steps: {step_count}, "
                          f"Final portfolio: ${info['portfolio_value']:.2f}")
        except Exception as e:
            print(f"Error during training stage {stage+1}: {e}")
            print("Attempting to continue with next stage...")

    # Save the final trained model
    try:
        model.save(model_save_path)
        print(f"Training completed at {datetime.now()} and model saved to {model_save_path}!")
    except Exception as e:
        print(f"Error saving model: {e}")
        print("Training completed but model could not be saved.")

    return model

# Calculate common portfolio performance metrics

In [109]:
def calculate_portfolio_metrics(portfolio_values, risk_free_rate=0.02):
    """Calculate common portfolio performance metrics"""
    if len(portfolio_values) < 2:
        return {
            "total_return": 0,
            "sharpe_ratio": 0,
            "max_drawdown": 0,
            "volatility": 0
        }

    # Calculate returns
    returns = np.array([(portfolio_values[i] / portfolio_values[i-1]) - 1
                       for i in range(1, len(portfolio_values))])

    # Calculate metrics
    total_return = (portfolio_values[-1] / portfolio_values[0]) - 1
    daily_returns_mean = np.mean(returns)
    daily_returns_std = np.std(returns)

    # Annualize (assuming 252 trading days)
    annualized_return = ((1 + daily_returns_mean) ** 252) - 1
    annualized_vol = daily_returns_std * np.sqrt(252)

    # Sharpe ratio
    sharpe_ratio = (annualized_return - risk_free_rate) / annualized_vol if annualized_vol > 0 else 0

    # Maximum drawdown
    peak = portfolio_values[0]
    max_drawdown = 0

    for value in portfolio_values:
        if value > peak:
            peak = value
        drawdown = (peak - value) / peak
        max_drawdown = max(max_drawdown, drawdown)

    return {
        "total_return": total_return * 100,  # Convert to percentage
        "annualized_return": annualized_return * 100,
        "volatility": annualized_vol * 100,
        "sharpe_ratio": sharpe_ratio,
        "max_drawdown": max_drawdown * 100,
        "win_rate": np.mean(returns > 0) * 100
    }



# Backtest of the Portfolio Strategy

In [110]:
def backtest_portfolio(model, data_path, max_stocks=100, lookback=30, initial_cash=10000, years=1):
    """Run a comprehensive backtest of the portfolio strategy"""
    print(f"Running {years}-year backtest simulation...")

    # Initialize the environment with the backtest data
    backtest_iter = StockPortfolioIterator(
        data_path,
        lookback_window=lookback,
        min_history=100,
        max_stocks=max_stocks,
        train_years=years
    )

    backtest_env = PortfolioTradingEnv(backtest_iter, initial_cash=initial_cash)

    # Run backtest
    obs, _ = backtest_env.reset()
    done = False

    portfolio_values = [initial_cash]
    returns = []
    cash_allocations = []
    transaction_costs = []
    holdings_diversification = []
    daily_turnover = []

    step = 0
    while not done:
        action, _ = model.predict(obs, deterministic=True)

        # Track portfolio composition
        cash_allocations.append(action[0])

        # Number of stocks with allocation > 1%
        significant_holdings = sum(1 for a in action[1:] if a > 0.01)
        holdings_diversification.append(significant_holdings)

        # Execute step
        obs, reward, done, _, info = backtest_env.step(action)

        # Track metrics
        portfolio_values.append(info["portfolio_value"])
        if step > 0:
            returns.append((portfolio_values[-1] / portfolio_values[-2]) - 1)

        # Handle missing 'transaction_costs' safely
        transaction_costs.append(info.get("transaction_costs", 0))

        # Calculate turnover (sum of absolute changes in allocation)
        if step == 0:
            daily_turnover.append(0)
        else:
            turnover = info.get("transaction_costs", 0) / (portfolio_values[-2] * backtest_env.transaction_cost)
            daily_turnover.append(turnover)

        step += 1

        # Print progress
        if step % 50 == 0:
            print(f"Backtest step {step}, Portfolio value: ${portfolio_values[-1]:.2f}")

    # Calculate performance metrics
    metrics = calculate_portfolio_metrics(portfolio_values)

    backtest_results = {
        "portfolio_values": portfolio_values,
        "returns": returns,
        "cash_allocations": cash_allocations,
        "transaction_costs": transaction_costs,
        "holdings_diversification": holdings_diversification,
        "daily_turnover": daily_turnover,
        "metrics": metrics,
        "steps": step,
        "final_value": portfolio_values[-1]
    }

    return backtest_results


In [111]:
import sys
import os
import json
import numpy as np
from datetime import datetime
import logging
from stable_baselines3 import A2C

# Logging setup
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Default parameters
DATA_PATH = "/content/historical_data.csv"
MODEL_SAVE_PATH = "/content/drive/MyDrive/a2c_portfolio_model"
MAX_STOCKS = 100
LOOKBACK = 30
TIMESTEPS = 20000
INITIAL_CASH = 10000
MODE = "train_and_evaluate"
RESULTS_PATH = "results"

# Create results directory
os.makedirs(RESULTS_PATH, exist_ok=True)

# Check if running in Jupyter/Colab environment
is_notebook = 'ipykernel' in sys.modules

if is_notebook:
    logging.info("Running in notebook environment. Using default parameters.")
else:
    import argparse
    parser = argparse.ArgumentParser(description="Portfolio Optimization with Reinforcement Learning")
    parser.add_argument("--data_path", type=str, default=DATA_PATH)
    parser.add_argument("--model_path", type=str, default=MODEL_SAVE_PATH)
    parser.add_argument("--max_stocks", type=int, default=MAX_STOCKS)
    parser.add_argument("--lookback", type=int, default=LOOKBACK)
    parser.add_argument("--timesteps", type=int, default=TIMESTEPS)
    parser.add_argument("--initial_cash", type=float, default=INITIAL_CASH)
    parser.add_argument("--mode", type=str, default=MODE, choices=["train", "evaluate", "backtest", "recommend", "train_and_evaluate"])
    parser.add_argument("--results_path", type=str, default=RESULTS_PATH)

    args = parser.parse_args()
    DATA_PATH = args.data_path
    MODEL_SAVE_PATH = args.model_path
    MAX_STOCKS = args.max_stocks
    LOOKBACK = args.lookback
    TIMESTEPS = args.timesteps
    INITIAL_CASH = args.initial_cash
    MODE = args.mode
    RESULTS_PATH = args.results_path

    os.makedirs(RESULTS_PATH, exist_ok=True)

# File checks
try:
    with open(DATA_PATH, 'r') as f:
        pass  # Confirm file exists
except FileNotFoundError:
    logging.error(f"Error: Data file '{DATA_PATH}' not found.")
    exit(1)
except PermissionError:
    logging.error(f"Error: Permission denied for file '{DATA_PATH}'.")
    exit(1)

# Main execution logic
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

if MODE == "train" or MODE == "train_and_evaluate":
    logging.info("Starting training...")
    model = train_model(DATA_PATH, MODEL_SAVE_PATH, MAX_STOCKS, LOOKBACK, TIMESTEPS)
    logging.info(f"Model saved to {MODEL_SAVE_PATH}")

if MODE != "train":
    model_loaded = False
    attempt = 0
    while not model_loaded and attempt < 3:
        try:
            logging.info(f"Loading model from {MODEL_SAVE_PATH} (Attempt {attempt + 1})")
            model = A2C.load(MODEL_SAVE_PATH)
            model_loaded = True
        except Exception as e:
            logging.error(f"Error loading model: {e}")
            if attempt == 2:
                logging.error("Maximum retries reached. Exiting...")
                exit(1)
            attempt += 1
            MODEL_SAVE_PATH = input("Enter a valid model path: ")

if MODE == "evaluate" or MODE == "train_and_evaluate":
    logging.info("Evaluating model...")
    rewards, values = evaluate_model(model, DATA_PATH, MAX_STOCKS, LOOKBACK, episodes=10)

    eval_results = {
        "rewards": [float(r) for r in rewards],
        "final_values": [float(v) for v in values],
        "avg_reward": float(np.mean(rewards)),
        "avg_final_value": float(np.mean(values)),
        "max_reward": float(np.max(rewards)),
        "min_reward": float(np.min(rewards)),
        "success_rate": float(np.mean(np.array(rewards) > 0) * 100)
    }

    with open(f"{RESULTS_PATH}/evaluation_{timestamp}.json", "w") as f:
        json.dump(eval_results, f, indent=2)

    logging.info(f"Evaluation results saved to {RESULTS_PATH}/evaluation_{timestamp}.json")

if MODE == "recommend":
    logging.info("Generating portfolio recommendation...")
    recommender = PortfolioRecommender(MODEL_SAVE_PATH, DATA_PATH, max_stocks=MAX_STOCKS, lookback=LOOKBACK)
    recommendation = recommender.recommend_portfolio(INITIAL_CASH)

    with open(f"{RESULTS_PATH}/recommendation_{timestamp}.json", "w") as f:
        json.dump(recommendation, f, indent=2)

    logging.info(f"Recommendation saved to {RESULTS_PATH}/recommendation_{timestamp}.json")

logging.info("Portfolio optimization complete. All results saved in the 'results' directory.")


Starting training at 2025-03-25 02:22:10.816591
Loading data from /content/historical_data.csv...
Adding technical indicators...
Technical indicators added successfully
Scaling features...
Preparing training data...


Output()

/usr/local/lib/python3.11/dist-packages/ipywidgets/widgets/widget_output.py:111: DeprecationWarning: 
Kernel._parent_header is deprecated in ipykernel 6. Use .get_parent()
  if ip and hasattr(ip, 'kernel') and hasattr(ip.kernel, '_parent_header'):

Loaded 100 stocks with 5021 trading days
Date range: 2004-12-30 00:00:00 to 2024-12-30 00:00:00
Evaluating random policy before training...
Random policy evaluation: Total steps: 100, Total reward: -0.13316288590431213
Training model with 100 stocks, each with 30 days of history

Training stage 1/5 (4000 timesteps)...


Checkpoint saved to /content/drive/MyDrive/a2c_portfolio_model_checkpoint_1
Quick evaluation of current policy...
  Episode 1 reward: -0.0811, Steps: 100, Final portfolio: $9626.23
  Episode 2 reward: -0.0811, Steps: 100, Final portfolio: $9626.23


...

  Episode 3 reward: -0.0711, Steps: 100, Final portfolio: $9723.47

Training stage 4/5 (4000 timesteps)...


Checkpoint saved to /content/drive/MyDrive/a2c_portfolio_model_checkpoint_4
Quick evaluation of current policy...
  Episode 1 reward: -0.0740, Steps: 100, Final portfolio: $9709.80


Output()

  Episode 2 reward: -0.0740, Steps: 100, Final portfolio: $9709.80
  Episode 3 reward: -0.0740, Steps: 100, Final portfolio: $9709.80

Training stage 5/5 (4000 timesteps)...


Checkpoint saved to /content/drive/MyDrive/a2c_portfolio_model_checkpoint_5
Training completed at 2025-03-25 02:28:47.292460 and model saved to /content/drive/MyDrive/a2c_portfolio_model!
Evaluating model performance...
Loading data from /content/historical_data.csv...
Adding technical indicators...
Technical indicators added successfully
Scaling features...
Preparing training data...
Loaded 100 stocks with 1256 trading days
Date range: 2019-12-30 00:00:00 to 2024-12-30 00:00:00
Episode 1: Return = 0.4537, Steps = 1226, Final Value = $19608.84, Annualized Return = 14.85%
Episode 2: Return = 0.4537, Steps = 1226, Final Value = $19608.84, Annualized Return = 14.85%
Episode 3: Return = 0.4537, Steps = 1226, Final Value = $19608.84, Annualized Return = 14.85%
Episode 4: Return = 0.4537, Steps = 1226, Final Value = $19608.84, Annualized Return = 14.85%
Episode 5: Return = 0.4537, Steps = 1226, Final Value = $19608.84, Annualized Return = 14.85%
Episode 6: Return = 0.4537, Steps = 1226, Fina

In [112]:
import os
import json
from datetime import datetime

# Ensure the results directory exists
os.makedirs("results", exist_ok=True)

# Change the mode to 'recommend'
MODE = "recommend"

# Optionally, you can modify the initial cash amount
INITIAL_CASH = 10000  # or any other amount you prefer

# Define a threshold for significant allocations
SIGNIFICANT_ALLOCATION_THRESHOLD = 1.0

# Run the recommendation part of the code
if MODE == "recommend":
    print(f"\n{'='*80}\nGENERATING PORTFOLIO RECOMMENDATION\n{'='*80}")
    try:
        # Generate portfolio recommendation
        recommender = PortfolioRecommender(
            MODEL_SAVE_PATH,
            DATA_PATH,
            max_stocks=MAX_STOCKS,
            lookback=LOOKBACK
        )

        recommendation = recommender.recommend_portfolio(INITIAL_CASH)

        # Print recommendation
        print(f"Recommended portfolio allocation (as of {recommendation['recommendation_date']}):")
        print(f"Cash: ${recommendation['cash_amount']:.2f} ({recommendation['cash_percent']:.2f}%)")
        print("\nStock allocations:")

        # Sort allocations by percentage (largest first)
        sorted_allocations = sorted(
            recommendation['stock_allocations'].items(),
            key=lambda x: x[1]['allocation_percent'],
            reverse=True
        )

        for ticker, details in sorted_allocations:
            if details['allocation_percent'] > SIGNIFICANT_ALLOCATION_THRESHOLD:
                print(f"{ticker}: ${details['allocation_cad']:.2f} "
                      f"({details['allocation_percent']:.2f}%) - "
                      f"{details['shares']:.4f} shares @ ${details['price_per_share']:.2f}")

        # Calculate some portfolio statistics
        if sorted_allocations:
            allocations = [details['allocation_percent'] for _, details in sorted_allocations]
            print("\nPortfolio allocation statistics:")
            print(f"Number of stocks: {len(allocations)}")
            print(f"Max allocation: {max(allocations):.2f}%")
            print(f"Min allocation: {min(allocations):.2f}%")
            print(f"Avg allocation: {sum(allocations)/len(allocations):.2f}%")
            print(f"Diversification score: {len([a for a in allocations if a > SIGNIFICANT_ALLOCATION_THRESHOLD])}")

        # Save recommendation
        recommendation_path = f"results/recommendation_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
        with open(recommendation_path, "w") as f:
            json.dump(recommendation, f, indent=2)

        print(f"Recommendation saved to {recommendation_path}")

    except Exception as e:
        print(f"Error during recommendation generation: {e}")



GENERATING PORTFOLIO RECOMMENDATION
Recommended portfolio allocation (as of 2024-12-30 00:00:00):
Cash: $0.00 (0.00%)

Stock allocations:
HQU.TO: $486.91 (4.87%) - 20.1285 shares @ $24.19
XEG.TO: $410.35 (4.10%) - 24.3818 shares @ $16.83
DOL.TO: $364.79 (3.65%) - 2.6107 shares @ $139.73
SU.TO: $359.12 (3.59%) - 7.0818 shares @ $50.71
VET.TO: $358.90 (3.59%) - 27.8645 shares @ $12.88
ATD.TO: $354.54 (3.55%) - 4.4805 shares @ $79.13
TWM.TO: $347.90 (3.48%) - 2676.1260 shares @ $0.13
PXT.TO: $342.62 (3.43%) - 25.0633 shares @ $13.67
BTCC.TO: $339.82 (3.40%) - 19.4407 shares @ $17.48
FRU.TO: $335.41 (3.35%) - 26.5986 shares @ $12.61
ATH.TO: $329.63 (3.30%) - 64.0053 shares @ $5.15
K.TO: $308.87 (3.09%) - 23.4524 shares @ $13.17
CXB.TO: $297.53 (2.98%) - 141.6820 shares @ $2.10
IMG.TO: $290.18 (2.90%) - 39.9151 shares @ $7.27
IMO.TO: $287.30 (2.87%) - 3.2722 shares @ $87.80
CNR.TO: $278.17 (2.78%) - 1.9216 shares @ $144.76
H.TO: $273.87 (2.74%) - 6.1822 shares @ $44.30
NGD.TO: $252.37 (2.5

In [113]:
# Change the mode to 'backtest'
MODE = "backtest"

# Optionally, customize these parameters
INITIAL_CASH = 10000  # Starting portfolio value
LOOKBACK = 30         # Days of history to consider
MAX_STOCKS = 100      # Maximum number of stocks to consider

# Run the backtest
if MODE == "backtest":
    print(f"\n{'='*80}\nRUNNING BACKTEST\n{'='*80}")
    # Run comprehensive backtest
    backtest_results = backtest_portfolio(
        model,
        DATA_PATH,
        MAX_STOCKS,
        LOOKBACK,
        INITIAL_CASH,
        years=2  # Use 2 years of data for backtest
    )

    # Print backtest summary
    metrics = backtest_results["metrics"]
    print("\nBacktest Summary:")
    print(f"Total Return: {metrics['total_return']:.2f}%")
    print(f"Annualized Return: {metrics['annualized_return']:.2f}%")
    print(f"Volatility: {metrics['volatility']:.2f}%")
    print(f"Sharpe Ratio: {metrics['sharpe_ratio']:.2f}")
    print(f"Maximum Drawdown: {metrics['max_drawdown']:.2f}%")
    print(f"Win Rate: {metrics['win_rate']:.2f}%")
    print(f"Average Cash Allocation: {np.mean(backtest_results['cash_allocations']) * 100:.2f}%")
    print(f"Average Number of Holdings: {np.mean(backtest_results['holdings_diversification']):.1f} stocks")
    print(f"Average Daily Turnover: {np.mean(backtest_results['daily_turnover']) * 100:.2f}%")

    # Save backtest results (excluding large arrays)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    backtest_summary = {
        "initial_cash": INITIAL_CASH,
        "final_value": float(backtest_results["final_value"]),
        "trading_days": backtest_results["steps"],
        "metrics": {k: float(v) for k, v in metrics.items()},
        "avg_cash_allocation": float(np.mean(backtest_results["cash_allocations"]) * 100),
        "avg_holdings": float(np.mean(backtest_results["holdings_diversification"])),
        "avg_turnover": float(np.mean(backtest_results["daily_turnover"]) * 100),
        "avg_transaction_costs": float(np.mean(backtest_results["transaction_costs"]))
    }

    with open(f"results/backtest_{timestamp}.json", "w") as f:
        import json
        json.dump(backtest_summary, f, indent=2)

    print(f"Backtest summary saved to results/backtest_{timestamp}.json")


RUNNING BACKTEST
Running 2-year backtest simulation...
Loading data from /content/historical_data.csv...
Adding technical indicators...
Technical indicators added successfully
Scaling features...
Preparing training data...
Loaded 100 stocks with 502 trading days
Date range: 2022-12-30 00:00:00 to 2024-12-30 00:00:00
Backtest step 50, Portfolio value: $9848.74
Backtest step 100, Portfolio value: $9369.07
Backtest step 150, Portfolio value: $9557.96
Backtest step 200, Portfolio value: $9464.59
Backtest step 250, Portfolio value: $9562.24
Backtest step 300, Portfolio value: $10345.09
Backtest step 350, Portfolio value: $10539.66
Backtest step 400, Portfolio value: $10332.59
Backtest step 450, Portfolio value: $10790.40

Backtest Summary:
Total Return: 2.55%
Annualized Return: 2.61%
Volatility: 15.66%
Sharpe Ratio: 0.04
Maximum Drawdown: 12.92%
Win Rate: 52.75%
Average Cash Allocation: 36.45%
Average Number of Holdings: 47.5 stocks
Average Daily Turnover: 0.00%
Backtest summary saved to r

In [115]:
import itertools
import time
from tqdm import tqdm


def hyperparameter_tuning(
    data_path,
    base_model_path="tuning_models",
    max_stocks_options=[50, 100],
    lookback_options=[15, 30, 45],
    learning_rate_options=[1e-4, 3e-4, 1e-3],whatsa
    n_steps_options=[512, 1024],
    batch_size_options=[64, 128],
    transaction_cost_options=[0.001, 0.002],
    n_epochs_options=[5, 10],
    timesteps=100000,  # Reduced for tuning
    eval_episodes=3,   # Reduced for tuning
    n_eval_envs=2
):
    """
    Perform hyperparameter tuning for the portfolio optimization model.

    Parameters:
    -----------
    data_path : str
        Path to the historical data CSV
    base_model_path : str
        Base path to save trained models
    max_stocks_options : list
        Different numbers of stocks to consider
    lookback_options : list
        Different lookback window sizes
    learning_rate_options : list
        Different learning rates for A2C
    gamma_options : list
        Different discount factors
    ent_coef_options : list
        Different entropy coefficients for exploration
    n_steps_options : list
        Different step sizes for rollout
    batch_size_options : list
        Different batch sizes for training
    transaction_cost_options : list
        Different transaction cost values
    n_epochs_options : list
        Different numbers of optimization epochs
    timesteps : int
        Number of timesteps for each training run
    eval_episodes : int
        Number of episodes for evaluation
    n_eval_envs : int
        Number of environments for parallel evaluation

    Returns:
    --------
    dict
        Results of hyperparameter tuning
    """
    # Create results directory
    os.makedirs(os.path.join(base_model_path, "results"), exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    tuning_results_path = os.path.join(base_model_path, "results", f"tuning_results_{timestamp}.json")

    # Track all results
    all_results = []

    # Generate parameter grid (selecting key parameters to avoid combinatorial explosion)
    parameter_grid = []
    for max_stocks, lookback, lr, gamma, ent_coef in itertools.product(
        max_stocks_options,
        lookback_options,
        learning_rate_options,
        gamma_options,
        ent_coef_options
    ):
        # Fixed parameters for initial tuning round
        n_steps = n_steps_options[0]
        batch_size = batch_size_options[0]
        transaction_cost = transaction_cost_options[0]
        n_epochs = n_epochs_options[0]

        parameter_grid.append({
            "max_stocks": max_stocks,
            "lookback": lookback,
            "learning_rate": lr,
            "gamma": gamma,
            "ent_coef": ent_coef,
            "n_steps": n_steps,
            "batch_size": batch_size,
            "transaction_cost": transaction_cost,
            "n_epochs": n_epochs
        })

    print(f"Tuning with {len(parameter_grid)} parameter combinations")

    # Run tuning
    for i, params in enumerate(tqdm(parameter_grid, desc="Hyperparameter tuning")):
        start_time = time.time()

        try:
            # Extract parameters
            max_stocks = params["max_stocks"]
            lookback = params["lookback"]
            learning_rate = params["learning_rate"]
            gamma = params["gamma"]
            ent_coef = params["ent_coef"]
            n_steps = params["n_steps"]
            batch_size = params["batch_size"]
            transaction_cost = params["transaction_cost"]
            n_epochs = params["n_epochs"]

            model_name = f"model_{i+1}_ms{max_stocks}_lb{lookback}_lr{learning_rate}_g{gamma}_e{ent_coef}"
            model_path = os.path.join(base_model_path, model_name)

            print(f"\nTraining model {i+1}/{len(parameter_grid)}: {model_name}")

            # Initialize data iterator
            train_iter = StockPortfolioIterator(
                data_path,
                lookback_window=lookback,
                min_history=100,
                max_stocks=max_stocks,
                train_years=10  # Reduced for faster tuning
            )

            # Create environment
            def make_env(transaction_cost=transaction_cost):
                return PortfolioTradingEnv(train_iter.reset(), transaction_cost=transaction_cost)

            train_env = DummyVecEnv([make_env])

            # Create model
            model = A2C(
                "MlpPolicy",
                train_env,
                learning_rate=learning_rate,
                n_steps=n_steps,
                batch_size=batch_size,
                n_epochs=n_epochs,
                gamma=gamma,
                ent_coef=ent_coef,
                verbose=0,
                device="auto"
            )

            # Train model with reduced timesteps for tuning
            model.learn(total_timesteps=timesteps, progress_bar=True)

            # Evaluate model
            rewards, values = evaluate_model(
                model,
                data_path,
                max_stocks,
                lookback,
                episodes=eval_episodes,
                verbose=False
            )

            # Run backtest
            backtest_results = backtest_portfolio(
                model,
                data_path,
                max_stocks,
                lookback,
                initial_cash=10000,
                years=1,  # Reduced for faster tuning
                verbose=False
            )

            # Calculate metrics
            train_time = time.time() - start_time
            avg_reward = np.mean(rewards)
            avg_final_value = np.mean(values)
            success_rate = np.mean(np.array(rewards) > 0) * 100

            # Extract key backtest metrics
            backtest_metrics = backtest_results["metrics"]

            # Store results
            result = {
                "model_name": model_name,
                "parameters": params,
                "training_time": train_time,
                "evaluation": {
                    "avg_reward": float(avg_reward),
                    "avg_final_value": float(avg_final_value),
                    "success_rate": float(success_rate)
                },
                "backtest": {
                    "total_return": float(backtest_metrics["total_return"]),
                    "annualized_return": float(backtest_metrics["annualized_return"]),
                    "volatility": float(backtest_metrics["volatility"]),
                    "sharpe_ratio": float(backtest_metrics["sharpe_ratio"]),
                    "max_drawdown": float(backtest_metrics["max_drawdown"]),
                    "win_rate": float(backtest_metrics["win_rate"])
                },
                "portfolio_characteristics": {
                    "avg_cash_allocation": float(np.mean(backtest_results["cash_allocations"]) * 100),
                    "avg_holdings": float(np.mean(backtest_results["holdings_diversification"])),
                    "avg_turnover": float(np.mean(backtest_results["daily_turnover"]) * 100)
                }
            }

            all_results.append(result)

            # Save model
            model.save(model_path)

            # Save interim results
            with open(tuning_results_path, "w") as f:
                json.dump(all_results, f, indent=2)

            print(f"Model {i+1} results:")
            print(f"  Avg reward: {avg_reward:.4f}")
            print(f"  Avg final value: ${avg_final_value:.2f}")
            print(f"  Success rate: {success_rate:.2f}%")
            print(f"  Sharpe ratio: {backtest_metrics['sharpe_ratio']:.2f}")
            print(f"  Annualized return: {backtest_metrics['annualized_return']:.2f}%")
            print(f"  Max drawdown: {backtest_metrics['max_drawdown']:.2f}%")

        except Exception as e:
            print(f"Error in parameter set {i+1}: {e}")
            all_results.append({
                "model_name": f"model_{i+1}",
                "parameters": params,
                "error": str(e)
            })

    # Find best model by different metrics
    valid_results = [r for r in all_results if "error" not in r]

    if valid_results:
        best_by_reward = max(valid_results, key=lambda x: x["evaluation"]["avg_reward"])
        best_by_sharpe = max(valid_results, key=lambda x: x["backtest"]["sharpe_ratio"])
        best_by_return = max(valid_results, key=lambda x: x["backtest"]["annualized_return"])

        best_models = {
            "best_by_reward": best_by_reward,
            "best_by_sharpe": best_by_sharpe,
            "best_by_return": best_by_return
        }

        # Save final results
        final_results = {
            "timestamp": timestamp,
            "all_results": all_results,
            "best_models": best_models
        }

        with open(tuning_results_path, "w") as f:
            json.dump(final_results, f, indent=2)

        print("\nHyperparameter tuning complete!")
        print(f"Results saved to: {tuning_results_path}")

        print("\nBest model by evaluation reward:")
        print_model_summary(best_by_reward)

        print("\nBest model by Sharpe ratio:")
        print_model_summary(best_by_sharpe)

        print("\nBest model by annualized return:")
        print_model_summary(best_by_return)

        return final_results
    else:
        print("No valid results found.")
        return {"timestamp": timestamp, "all_results": all_results}


def print_model_summary(model_result):
    """Print a summary of model performance"""
    params = model_result["parameters"]
    eval_metrics = model_result["evaluation"]
    backtest_metrics = model_result["backtest"]
    portfolio_metrics = model_result["portfolio_characteristics"]

    print(f"Model: {model_result['model_name']}")
    print(f"Parameters: max_stocks={params['max_stocks']}, lookback={params['lookback']}, "
          f"lr={params['learning_rate']}, gamma={params['gamma']}, ent_coef={params['ent_coef']}")
    print(f"Evaluation: reward={eval_metrics['avg_reward']:.4f}, final_value=${eval_metrics['avg_final_value']:.2f}")
    print(f"Backtest: return={backtest_metrics['annualized_return']:.2f}%, "
          f"sharpe={backtest_metrics['sharpe_ratio']:.2f}, "
          f"drawdown={backtest_metrics['max_drawdown']:.2f}%")
    print(f"Portfolio: cash={portfolio_metrics['avg_cash_allocation']:.2f}%, "
          f"holdings={portfolio_metrics['avg_holdings']:.1f}, "
          f"turnover={portfolio_metrics['avg_turnover']:.2f}%")


def fine_tune_best_model(
    data_path,
    base_results_path,
    best_model_type="best_by_sharpe",  # Options: best_by_reward, best_by_sharpe, best_by_return
    timesteps=500000,  # Full training for fine-tuning
    eval_episodes=10
):
    """
    Fine-tune the best model from initial hyperparameter tuning

    Parameters:
    -----------
    data_path : str
        Path to the historical data CSV
    base_results_path : str
        Path to the tuning results JSON file
    best_model_type : str
        Which "best" model to fine-tune
    timesteps : int
        Number of timesteps for fine-tuning
    eval_episodes : int
        Number of episodes for evaluation

    Returns:
    --------
    dict
        Results of fine-tuning
    """
    # Load tuning results
    with open(base_results_path, "r") as f:
        tuning_results = json.load(f)

    best_model = tuning_results["best_models"][best_model_type]
    best_params = best_model["parameters"]

    print(f"Fine-tuning the {best_model_type} model:")
    print_model_summary(best_model)

    # Extract parameters
    max_stocks = best_params["max_stocks"]
    lookback = best_params["lookback"]
    learning_rate = best_params["learning_rate"]
    gamma = best_params["gamma"]
    ent_coef = best_params["ent_coef"]
    n_steps = best_params["n_steps"]
    batch_size = best_params["batch_size"]
    transaction_cost = best_params["transaction_cost"]
    n_epochs = best_params["n_epochs"]

    # Create a timestamp for this fine-tuning run
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    model_name = f"finetuned_{best_model_type}_{timestamp}"
    model_path = os.path.join(os.path.dirname(base_results_path), model_name)

    # Initialize data iterator with full data
    train_iter = StockPortfolioIterator(
        data_path,
        lookback_window=lookback,
        min_history=100,
        max_stocks=max_stocks,
        train_years=20  # Full training data
    )

    # Create environment
    def make_env(transaction_cost=transaction_cost):
        return PortfolioTradingEnv(train_iter.reset(), transaction_cost=transaction_cost)

    train_env = DummyVecEnv([make_env])

    # Create model
    model = A2C(
        "MlpPolicy",
        train_env,
        learning_rate=learning_rate,
        n_steps=n_steps,
        batch_size=batch_size,
        n_epochs=n_epochs,
        gamma=gamma,
        ent_coef=ent_coef,
        verbose=1,
        device="auto",
        tensorboard_log=os.path.join(os.path.dirname(base_results_path), "fine_tuning_tensorboard")
    )

    print(f"\nFine-tuning with {timesteps} timesteps...")

    # Train model with full timesteps
    model.learn(total_timesteps=timesteps, progress_bar=True)
    model.save(model_path)

    print(f"Model saved to {model_path}")

    # Evaluate model
    print("\nEvaluating fine-tuned model...")
    rewards, values = evaluate_model(
        model,
        data_path,
        max_stocks,
        lookback,
        episodes=eval_episodes
    )

    # Run backtest
    print("\nRunning backtest with fine-tuned model...")
    backtest_results = backtest_portfolio(
        model,
        data_path,
        max_stocks,
        lookback,
        initial_cash=10000,
        years=2  # Full backtest
    )

    # Calculate metrics
    avg_reward = np.mean(rewards)
    avg_final_value = np.mean(values)
    success_rate = np.mean(np.array(rewards) > 0) * 100

    # Extract key backtest metrics
    backtest_metrics = backtest_results["metrics"]

    # Store results
    fine_tuning_result = {
        "model_name": model_name,
        "model_path": model_path,
        "original_model": best_model["model_name"],
        "parameters": best_params,
        "evaluation": {
            "avg_reward": float(avg_reward),
            "avg_final_value": float(avg_final_value),
            "success_rate": float(success_rate)
        },
        "backtest": {
            "total_return": float(backtest_metrics["total_return"]),
            "annualized_return": float(backtest_metrics["annualized_return"]),
            "volatility": float(backtest_metrics["volatility"]),
            "sharpe_ratio": float(backtest_metrics["sharpe_ratio"]),
            "max_drawdown": float(backtest_metrics["max_drawdown"]),
            "win_rate": float(backtest_metrics["win_rate"])
        },
        "portfolio_characteristics": {
            "avg_cash_allocation": float(np.mean(backtest_results["cash_allocations"]) * 100),
            "avg_holdings": float(np.mean(backtest_results["holdings_diversification"])),
            "avg_turnover": float(np.mean(backtest_results["daily_turnover"]) * 100)
        }
    }

    # Save results
    results_path = os.path.join(os.path.dirname(base_results_path), f"fine_tuning_results_{timestamp}.json")
    with open(results_path, "w") as f:
        json.dump(fine_tuning_result, f, indent=2)

    print(f"\nFine-tuning results saved to: {results_path}")
    print("\nFine-tuned model performance:")
    print(f"  Avg reward: {avg_reward:.4f}")
    print(f"  Avg final value: ${avg_final_value:.2f}")
    print(f"  Success rate: {success_rate:.2f}%")
    print(f"  Sharpe ratio: {backtest_metrics['sharpe_ratio']:.2f}")
    print(f"  Annualized return: {backtest_metrics['annualized_return']:.2f}%")
    print(f"  Max drawdown: {backtest_metrics['max_drawdown']:.2f}%")

    return fine_tuning_result


# Modified versions of evaluate_model and backtest_portfolio for tuning
def evaluate_model(model, data_path, max_stocks=100, lookback=30, episodes=10, verbose=True):
    """Evaluate the model on a validation dataset with option to silence output"""
    if verbose:
        print(f"Evaluating model performance...")

    eval_iter = StockPortfolioIterator(
        data_path,
        lookback_window=lookback,
        min_history=100,
        max_stocks=max_stocks,
        train_years=5  # Use more recent data for evaluation
    )

    eval_env = PortfolioTradingEnv(eval_iter)

    total_rewards = []
    portfolio_values = []

    for ep in range(episodes):
        obs, _ = eval_env.reset()
        done = False
        episode_reward = 0
        initial_value = eval_env.initial_cash
        step_count = 0

        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, done, _, info = eval_env.step(action)
            episode_reward += reward
            step_count += 1

            if done:
                final_value = info["portfolio_value"]
                portfolio_values.append(final_value)

                # Calculate annualized return
                if step_count > 0:
                    days = step_count
                    annualized_return = ((final_value / initial_value) ** (252 / days) - 1) * 100
                else:
                    annualized_return = 0

                if verbose:
                    print(f"Episode {ep+1}: Return = {episode_reward:.4f}, "
                          f"Steps = {step_count}, "
                          f"Final Value = ${final_value:.2f}, "
                          f"Annualized Return = {annualized_return:.2f}%")

        total_rewards.append(episode_reward)

    # Calculate summary statistics
    avg_reward = np.mean(total_rewards)
    avg_final_value = np.mean(portfolio_values)
    success_rate = np.sum(np.array(total_rewards) > 0) / len(total_rewards) * 100

    if verbose:
        print(f"\nEvaluation Results (over {episodes} episodes):")
        print(f"Average Total Reward: {avg_reward:.4f}")
        print(f"Average Final Portfolio Value: ${avg_final_value:.2f}")
        print(f"Success Rate (positive return): {success_rate:.2f}%")

    return total_rewards, portfolio_values


def backtest_portfolio(model, data_path, max_stocks=100, lookback=30, initial_cash=10000, years=2, verbose=True):
    """Run a comprehensive backtest of the portfolio strategy with option to silence output"""
    if verbose:
        print(f"Running {years}-year backtest simulation...")

    # Initialize the environment with the backtest data
    backtest_iter = StockPortfolioIterator(
        data_path,
        lookback_window=lookback,
        min_history=100,
        max_stocks=max_stocks,
        train_years=years
    )

    backtest_env = PortfolioTradingEnv(backtest_iter, initial_cash=initial_cash)

    # Run backtest
    obs, _ = backtest_env.reset()
    done = False

    portfolio_values = [initial_cash]
    returns = []
    cash_allocations = []
    transaction_costs = []
    holdings_diversification = []
    daily_turnover = []

    step = 0
    while not done:
        action, _ = model.predict(obs, deterministic=True)

        # Track portfolio composition
        cash_allocations.append(action[0])

        # Number of stocks with allocation > 1%
        significant_holdings = sum(1 for a in action[1:] if a > 0.01)
        holdings_diversification.append(significant_holdings)

        # Execute step
        obs, reward, done, _, info = backtest_env.step(action)

        # Track metrics
        portfolio_values.append(info["portfolio_value"])
        if step > 0:
            returns.append((portfolio_values[-1] / portfolio_values[-2]) - 1)
        transaction_costs.append(info["transaction_costs"])

        # Calculate turnover (sum of absolute changes in allocation)
        if step == 0:
            daily_turnover.append(0)
        else:
            turnover = info["transaction_costs"] / (portfolio_values[-2] * backtest_env.transaction_cost)
            daily_turnover.append(turnover)

        step += 1

        # Print progress
        if verbose and step % 50 == 0:
            print(f"Backtest step {step}, Portfolio value: ${portfolio_values[-1]:.2f}")

    # Calculate performance metrics
    metrics = calculate_portfolio_metrics(portfolio_values)

    backtest_results = {
        "portfolio_values": portfolio_values,
        "returns": returns,
        "cash_allocations": cash_allocations,
        "transaction_costs": transaction_costs,
        "holdings_diversification": holdings_diversification,
        "daily_turnover": daily_turnover,
        "metrics": metrics,
        "steps": step,
        "final_value": portfolio_values[-1]
    }

    return backtest_results


# Example usage:
if __name__ == "__main__":
    # Set up parameters for tuning
    DATA_PATH = "/content/historical_data.csv"
    TUNING_DIR = "tuning_models"

    # First round of hyperparameter tuning with reduced parameter space
    tuning_results = hyperparameter_tuning(
        DATA_PATH,
        base_model_path=TUNING_DIR,
        max_stocks_options=[50, 100],
        lookback_options=[15, 30],
        learning_rate_options=[1e-4, 3e-4],
        gamma_options=[0.99],
        ent_coef_options=[0.01],
        timesteps=100000,  # Reduced for initial tuning
        eval_episodes=3
    )

    # Get the path to the tuning results
    results_files = [f for f in os.listdir(os.path.join(TUNING_DIR, "results")) if f.startswith("tuning_results")]
    latest_results = sorted(results_files)[-1]
    results_path = os.path.join(TUNING_DIR, "results", latest_results)

    # Fine-tune the best model by Sharpe ratio
    fine_tuned_results = fine_tune_best_model(
        DATA_PATH,
        results_path,
        best_model_type="best_by_sharpe",
        timesteps=500000,  # Full training for the best model
        eval_episodes=10
    )

SyntaxError: invalid syntax (<ipython-input-115-39ef44f58a5c>, line 12)